# RGB + Depth + Thermal Segmentation Baseline

This notebook is a new multimodal research baseline that compares the existing RGB-only U-Net with a simple RGB + Depth + Thermal early-fusion model for BASEPROD terrain classification.

Important: this is a research prototype only, not a final uncertainty-aware system.
We reuse the exact same validation split and preprocessing conventions as the RGB baseline.


In [1]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = Path(r"C:\Users\Anushka\data\BASEPROD_labeled")
ZIP_PATH = Path(r"C:\Users\Anushka\data\BASEPROD_labeled.zip")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "multimodal_segmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATASET_ROOT={DATASET_ROOT}")
print(f"ZIP exists={ZIP_PATH.exists()}")
print(f"Device={DEVICE}")


DATASET_ROOT=C:\Users\Anushka\data\BASEPROD_labeled
ZIP exists=True
Device=cpu


In [3]:
rgb_dir = DATASET_ROOT / "BASEPROD" / "ORIGINAL" / "COLOR_IMGS"
mask_dir = DATASET_ROOT / "BASEPROD" / "ORIGINAL" / "MASK_RAWS"
rgb_files = sorted(rgb_dir.glob("*.png")) if rgb_dir.exists() else []
mask_files = sorted(mask_dir.glob("*_mask.png")) if mask_dir.exists() else []
paired = {p.stem: {"rgb": p, "mask": p.with_name(p.name + "_mask.png")} for p in rgb_files}
valid_keys = sorted(set(paired) & {m.name.replace("_mask.png", "") for m in mask_files})
combined = {k: {"rgb": paired[k]["rgb"], "mask": DATASET_ROOT / "BASEPROD" / "ORIGINAL" / "MASK_RAWS" / f"{k}_mask.png"} for k in valid_keys}
print("RGB files:", len(rgb_files))
print("Mask files:", len(mask_files))
print("Matched RGB-mask pairs:", len(combined))
print("Example keys:", sorted(list(combined.keys()))[:3])


RGB files: 950
Mask files: 950
Matched RGB-mask pairs: 950
Example keys: ['168986961442400000', '168986961982400000', '168986962142400000']


In [4]:
def read_csv_matrix(path):
    arr = np.genfromtxt(path, delimiter=",", dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr.astype(np.float32)

def depth_thermal_map_from_zip(zip_path):
    out = {"depth": {}, "thermal": {}}
    if not zip_path.exists():
        return out
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            low = name.lower()
            if low.endswith("d.csv") and "depth" in low:
                key = Path(name).stem.replace("d", "")
                out["depth"][key] = name
            if low.endswith("t.csv") and "thermal" in low:
                key = Path(name).stem.replace("t", "")
                out["thermal"][key] = name
    return out

zip_map = depth_thermal_map_from_zip(ZIP_PATH)
print("Depth entries in ZIP:", len(zip_map["depth"]))
print("Thermal entries in ZIP:", len(zip_map["thermal"]))
shared_keys = sorted(set(combined) & set(zip_map["depth"]) & set(zip_map["thermal"]))
print("Complete RGB+Depth+Thermal+Mask entries:", len(shared_keys))
print("Example complete key:", shared_keys[:3])


Depth entries in ZIP: 950
Thermal entries in ZIP: 950
Complete RGB+Depth+Thermal+Mask entries: 950
Example complete key: ['168986961442400000', '168986961982400000', '168986962142400000']


In [5]:
entries = []
for key in shared_keys:
    entries.append({
        "key": key,
        "rgb_path": str(combined[key]["rgb"]),
        "mask_path": str(combined[key]["mask"]),
        "depth_zip_name": zip_map["depth"][key],
        "thermal_zip_name": zip_map["thermal"][key],
    })

for e in entries[:3]:
    rgb = np.array(Image.open(e["rgb_path"]).convert("RGB"), dtype=np.uint8)
    mask = np.array(Image.open(e["mask_path"]), dtype=np.uint8)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        d = read_csv_matrix(zf.open(e["depth_zip_name"]))
        t = read_csv_matrix(zf.open(e["thermal_zip_name"]))
    print(e["key"], rgb.shape, mask.shape, d.shape, t.shape, np.unique(mask)[:10])
    if rgb.shape[:2] != (720, 1280):
        raise ValueError(f"Unexpected RGB shape for {e['key']}: {rgb.shape}")
    if mask.shape != (720, 1280):
        raise ValueError(f"Unexpected mask shape for {e['key']}: {mask.shape}")
    if d.shape != (720, 1280):
        raise ValueError(f"Unexpected depth shape for {e['key']}: {d.shape}")
    if t.shape != (720, 1280):
        raise ValueError(f"Unexpected thermal shape for {e['key']}: {t.shape}")
    if not np.isin(np.unique(mask), np.arange(8)).all():
        raise ValueError(f"Mask IDs out of range for {e['key']}: {np.unique(mask)}")
    if not np.isfinite(d).all() or not np.isfinite(t).all():
        raise ValueError(f"Non-finite values detected for {e['key']}")
print("Dataset validation: OK")


168986961442400000 (720, 1280, 3) (720, 1280) (720, 1280) (720, 1280) [0 1 2 5 6 7]
168986961982400000 (720, 1280, 3) (720, 1280) (720, 1280) (720, 1280) [0 1 2 5 6]
168986962142400000 (720, 1280, 3) (720, 1280) (720, 1280) (720, 1280) [0 1 2 6]
Dataset validation: OK


In [6]:
TARGET_SIZE = (256, 256)
CLASS_NAMES = ["Void", "Compact", "Grass", "Bedrock", "Sandy", "Pebble", "Rock", "Vegetation"]
rng = np.random.RandomState(SEED)
keys = np.array([e["key"] for e in entries])
rng.shuffle(keys)
val_size = max(1, int(round(len(keys) * 0.2)))
val_keys = set(keys[:val_size])
train_keys = set(keys[val_size:])
train_entries = [e for e in entries if e["key"] in train_keys]
val_entries = [e for e in entries if e["key"] in val_keys]
print("Train entries:", len(train_entries))
print("Validation entries:", len(val_entries))

class MultimodalDataset(Dataset):
    def __init__(self, entries, transform_rgb=None):
        self.entries = entries
        self.transform_rgb = transform_rgb or transforms.Compose([
            transforms.Resize(TARGET_SIZE, interpolation=InterpolationMode.BILINEAR),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, idx):
        e = self.entries[idx]
        rgb = Image.open(e["rgb_path"]).convert("RGB")
        mask = Image.open(e["mask_path"])
        with zipfile.ZipFile(ZIP_PATH) as zf:
            depth = np.genfromtxt(zf.open(e["depth_zip_name"]), delimiter=",", dtype=np.float32)
            thermal = np.genfromtxt(zf.open(e["thermal_zip_name"]), delimiter=",", dtype=np.float32)
        depth = np.asarray(depth, dtype=np.float32)
        thermal = np.asarray(thermal, dtype=np.float32)
        depth_img = Image.fromarray(depth.astype(np.float32))
        thermal_img = Image.fromarray(thermal.astype(np.float32))
        rgb_t = self.transform_rgb(rgb)
        depth_t = torch.tensor(np.asarray(depth_img.resize(TARGET_SIZE, resample=Image.BILINEAR), dtype=np.float32)).unsqueeze(0)
        thermal_t = torch.tensor(np.asarray(thermal_img.resize(TARGET_SIZE, resample=Image.BILINEAR), dtype=np.float32)).unsqueeze(0)
        mask_t = torch.tensor(np.asarray(Image.fromarray(np.array(mask)).resize(TARGET_SIZE, resample=Image.NEAREST), dtype=np.int64))
        x = torch.cat([rgb_t, depth_t / 255.0, thermal_t / 255.0], dim=0)
        return x, mask_t.long()

train_ds = MultimodalDataset(train_entries)
val_ds = MultimodalDataset(val_entries)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=0)
sample_x, sample_y = next(iter(train_loader))
print("Sample input shape:", tuple(sample_x.shape))
print("Sample label shape:", tuple(sample_y.shape))
print("Label IDs in sample:", torch.unique(sample_y).tolist())


Train entries: 760
Validation entries: 190
Sample input shape: (2, 5, 256, 256)
Sample label shape: (2, 256, 256)
Label IDs in sample: [0, 1, 2, 4, 5, 6, 7]


In [7]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.shape[2] - x1.shape[2]
        diffX = x2.shape[3] - x1.shape[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MultimodalUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=8):
        super().__init__()
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = Down(32, 64)
        self.down2 = Down(64, 128)
        self.down3 = Down(128, 256)
        self.down4 = Down(256, 512)
        self.up1 = Up(512, 256)
        self.up2 = Up(256, 128)
        self.up3 = Up(128, 64)
        self.up4 = Up(64, 32)
        self.out = nn.Conv2d(32, out_channels, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.out(x)

model = MultimodalUNet(in_channels=5, out_channels=8).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

smoke_x, smoke_y = sample_x, sample_y
smoke_x = smoke_x.to(DEVICE)
smoke_y = smoke_y.to(DEVICE).long()
smoke_logits = model(smoke_x)
smoke_loss = criterion(smoke_logits, smoke_y)
print("Smoke input shape:", tuple(smoke_x.shape))
print("Smoke mask shape:", tuple(smoke_y.shape))
print("Smoke output shape:", tuple(smoke_logits.shape))
print("Smoke loss finite:", torch.isfinite(smoke_loss).item(), float(smoke_loss.detach().cpu()))
smoke_loss.backward()
print("Gradients finite:", all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None))
before = {k: v.detach().clone() for k, v in model.state_dict().items()}
optimizer.step()
after = {k: v.detach().clone() for k, v in model.state_dict().items()}
print("Parameters changed after optimizer step:", any(not torch.allclose(before[k], after[k]) for k in before))
optimizer.zero_grad(set_to_none=True)


Smoke input shape: (2, 5, 256, 256)
Smoke mask shape: (2, 256, 256)
Smoke output shape: (2, 8, 256, 256)
Smoke loss finite: True 2.1398346424102783
Gradients finite: True
Parameters changed after optimizer step: True


In [8]:
def compute_metrics(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    confusion = torch.zeros((8, 8), dtype=torch.int64, device=device)
    total_batches = len(loader)
    progress_interval = max(1, total_batches // 10)

    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(loader, start=1):
            x = x.to(device)
            y = y.to(device).long()
            logits = model(x)
            loss = criterion(logits, y)
            pred = logits.argmax(dim=1)

            total_loss += loss.item() * x.shape[0]
            total_correct += (pred == y).sum().item()
            total_pixels += y.numel()
            encoded = (y.reshape(-1) * 8 + pred.reshape(-1)).long()
            confusion += torch.bincount(encoded, minlength=64).reshape(8, 8)

            if batch_idx % progress_interval == 0 or batch_idx == total_batches:
                print(f"Validation batches: {batch_idx}/{total_batches}")

    loss = total_loss / max(1, len(loader.dataset))
    acc = total_correct / max(1, total_pixels)
    cm = confusion.cpu().numpy().astype(np.float64)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    denom = tp + fp + fn
    iou = np.divide(tp, denom, out=np.zeros_like(tp), where=denom > 0)
    return float(loss), float(acc), float(iou.mean()), iou.astype(np.float32)


In [ ]:
history = []
best_val_miou = -1.0
best_state = None
for epoch in range(5):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_loss_sum = 0.0
    total_batches = len(train_loader)
    progress_interval = max(1, total_batches // 10)

    for batch_idx, (x, y) in enumerate(train_loader, start=1):
        x = x.to(DEVICE)
        y = y.to(DEVICE).long()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss_sum += loss.item() * x.shape[0]

        if batch_idx % progress_interval == 0 or batch_idx == total_batches:
            print(f"Epoch {epoch + 1}/5 | training batches: {batch_idx}/{total_batches} | batch_loss={loss.item():.4f}")

    train_loss = epoch_loss_sum / len(train_loader.dataset)
    val_loss, val_acc, val_miou, val_iou = compute_metrics(model, val_loader, criterion, DEVICE)
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_pixel_accuracy": val_acc,
        "val_miou": val_miou,
    })
    print(f"Epoch {epoch + 1:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_mIoU={val_miou:.4f}")
    if val_miou > best_val_miou:
        best_val_miou = val_miou
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
best_model_path = OUTPUT_DIR / "best_multimodal_unet.pt"
torch.save({"state_dict": model.state_dict(), "best_val_miou": best_val_miou}, best_model_path)
metrics_path = OUTPUT_DIR / "training_metrics.csv"
pd.DataFrame(history).to_csv(metrics_path, index=False)
print(f"Saved best model: {best_model_path}")
print(f"Saved training metrics: {metrics_path}")


Epoch 1/5 | training batches: 38/380 | batch_loss=1.8324
Epoch 1/5 | training batches: 76/380 | batch_loss=1.7385
Epoch 1/5 | training batches: 114/380 | batch_loss=1.5627
Epoch 1/5 | training batches: 152/380 | batch_loss=1.5096
Epoch 1/5 | training batches: 190/380 | batch_loss=1.6319
Epoch 1/5 | training batches: 228/380 | batch_loss=1.2050
Epoch 1/5 | training batches: 266/380 | batch_loss=1.5052
Epoch 1/5 | training batches: 304/380 | batch_loss=1.5204
Epoch 1/5 | training batches: 342/380 | batch_loss=1.3193
Epoch 1/5 | training batches: 380/380 | batch_loss=1.2729
Validation batches: 9/95
Validation batches: 18/95
Validation batches: 27/95
Validation batches: 36/95
Validation batches: 45/95
Validation batches: 54/95
Validation batches: 63/95
Validation batches: 72/95
Validation batches: 81/95
Validation batches: 90/95
Validation batches: 95/95
Epoch 01 | train_loss=1.5547 | val_loss=1.2729 | val_acc=0.6413 | val_mIoU=0.2558
Epoch 2/5 | training batches: 38/380 | batch_loss=1.349

In [ ]:
rgb_baseline_miou = 0.2862
rgb_baseline_acc = 0.668699
val_loss, val_acc, val_miou, val_iou = compute_metrics(model, val_loader, criterion, DEVICE)
print(f"Final val loss: {val_loss:.4f}")
print(f"Final val pixel accuracy: {val_acc:.4f}")
print(f"Final val mIoU: {val_miou:.4f}")
print("Per-class IoU:", val_iou)
print(f"mIoU improvement over RGB baseline: {val_miou - rgb_baseline_miou:.4f}")
print(f"Pixel accuracy improvement over RGB baseline: {val_acc - rgb_baseline_acc:.4f}")
pd.DataFrame({
    "class_id": list(range(8)),
    "class_name": CLASS_NAMES,
    "iou": val_iou,
}).to_csv(OUTPUT_DIR / "per_class_iou.csv", index=False)
final_eval = {
    "dataset_root": str(DATASET_ROOT),
    "train_samples": len(train_entries),
    "val_samples": len(val_entries),
    "input_channels": 5,
    "model": "EarlyFusionRGBDepthThermalUNet",
    "epochs": 5,
    "best_val_miou": float(val_miou),
    "pixel_accuracy": float(val_acc),
    "rgb_baseline_miou": float(rgb_baseline_miou),
    "rgb_baseline_pixel_accuracy": float(rgb_baseline_acc),
    "miou_improvement_over_rgb": float(val_miou - rgb_baseline_miou),
    "accuracy_improvement_over_rgb": float(val_acc - rgb_baseline_acc),
}
with open(OUTPUT_DIR / "final_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(final_eval, f, indent=2)
print(f"Saved final_evaluation.json: {OUTPUT_DIR / 'final_evaluation.json'}")
print(f"Saved per_class_iou.csv: {OUTPUT_DIR / 'per_class_iou.csv'}")


## Final summary

- Verified dataset pairing via filename matching and ZIP-based modality validation.
- Same split, resize policy, and evaluation conventions as the existing RGB baseline are reused.
- Model is a small CPU-compatible early-fusion RGB + Depth + Thermal U-Net.
- This notebook is a multimodal baseline experiment only; it does not claim geometric calibration or final rover-navigation readiness.
